# 🎙 자동 전사 (설정 불필요)

**사용법:**
1. 녹음 파일을 `내 드라이브/5.기타/_inbox/녹음/`에 업로드하세요.
2. **런타임 → 모두 실행** (Ctrl+F9) 하세요.

---
**작동 방식:**
- `output/` 폴더에 전사문이 없는 파일은 **무조건** 전사합니다.
- 중간에 멈췄다가 다시 실행해도 안전합니다 (이어하기).
- 파일명에 강사/강의/회차가 있으면 자동으로 인식해서 기록합니다.

In [1]:
#@title ⚙️ 기본 설정
MODEL_SIZE = "large-v3"  #@param ["large-v3", "large-v3-turbo", "medium", "small"]
LANGUAGE   = "ko"        #@param ["ko", "en", "ja"]

print(f"🤖 모델: {MODEL_SIZE} / 언어: {LANGUAGE}")

🤖 모델: large-v3 / 언어: ko


## 1️⃣ 환경 설정

In [2]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!pip install -q faster-whisper
print("\n✅ 설치 완료")

Tesla T4, 15360 MiB
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 65.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 81.4 MB/s eta 0:00:00

✅ 설치 완료


In [3]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive 마운트 완료")

Mounted at /content/drive
✅ Google Drive 마운트 완료


## 2️⃣ 모델 로드

In [4]:
from faster_whisper import WhisperModel
import time

print(f"⏳ 모델 로딩 중 ({MODEL_SIZE})...")
start = time.time()
model = WhisperModel(MODEL_SIZE, device="cuda", compute_type="float16")
print(f"✅ 모델 로드 완료 ({time.time()-start:.1f}초)")

⏳ 모델 로딩 중 (large-v3)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


✅ 모델 로드 완료 (33.5초)


## 3️⃣ 할 일 확인

In [5]:
import os, re
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive")
INPUT_DIR  = DRIVE_ROOT / "5.기타" / "_inbox" / "녹음"
OUTPUT_DIR = INPUT_DIR / "output"
DONE_DIR   = INPUT_DIR / "processed"
AUDIO_EXTS = {".mp3", ".wav", ".m4a", ".mp4", ".webm", ".flac", ".ogg", ".aac"}

for d in [INPUT_DIR, OUTPUT_DIR, DONE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def normalize_name(name):
    # 공백을 언더스코어로 바꾼 파일명 (저장 시 사용)
    return name.replace(' ', '_')

# 이미 존재하는 전사문 (파일명 매칭: 공백 ver, 언더스코어 ver 모두 체크)
existing = set()
for f in OUTPUT_DIR.iterdir():
    if f.name.endswith('_transcript.txt'):
        existing.add(f.name)

# 불완전 임시파일 정리
tmp_files = [f for f in OUTPUT_DIR.iterdir() if f.name.endswith('.tmp')]
if tmp_files:
    print(f"🗑 불완전 임시파일 {len(tmp_files)}개 정리 (다시 전사합니다)")
    for f in tmp_files:
        f.unlink()

# 전체 오디오 스캔
all_audio = [f for f in INPUT_DIR.iterdir() if f.is_file() and f.suffix.lower() in AUDIO_EXTS]

to_process = []
already_done = []

for f in sorted(all_audio, key=lambda x: x.name):
    # 이미 전사되었는지 확인
    out_s = f.stem + '_transcript.txt'
    out_u = normalize_name(f.stem) + '_transcript.txt'

    if out_s in existing or out_u in existing:
        already_done.append(f)
    else:
        to_process.append(f)

print(f"{'='*40}")
print(f"📊 현황")
print(f"{'='*40}")
print(f"  ✅ 완료: {len(already_done)}개")
print(f"  ▶ 전사 대기: {len(to_process)}개")
print(f"{'='*40}")

if to_process:
    mb = sum(f.stat().st_size for f in to_process) / (1024*1024)
    print(f"\n📦 미전사: {mb:.0f} MB / 예상 소요: ~{len(to_process)*3}분")
    print("\n📋 전사할 파일 목록:")
    for f in to_process[:10]:
        print(f"  • {f.name}")
    if len(to_process) > 10:
        print(f"  ... 외 {len(to_process)-10}개")
else:
    print(f"\n🎉 모든 파일 전사가 완료되었습니다!")

📊 현황
  ✅ 완료: 7개
  ▶ 전사 대기: 0개

🎉 모든 파일 전사가 완료되었습니다!


## 4️⃣ 전사 시작

In [6]:
import json
from datetime import datetime

def parse_filename_safe(filename):
    # 정보 추출 시도 (실패해도 None 리턴하고 계속 진행)
    stem = Path(filename).stem
    try:
        m = re.match(r'^(.+?)[\s_]+(.+?)[\s_]+(\d+)-(\S+)$', stem)
        if m:
            return {'instructor': m.group(1), 'course': m.group(2),
                    'lecture': int(m.group(3)), 'part': m.group(4)}
    except:
        pass
    return None

results = []
total = len(to_process)

if total == 0:
    print("✅ 전사할 파일이 없습니다.")
else:
    print(f"▶ {total}개 파일 전사 시작...\n")

    for idx, audio_path in enumerate(to_process, 1):
        print(f"{'='*60}")
        print(f"▶ [{idx}/{total}] {audio_path.name}")
        print(f"{'='*60}")

        out_name = normalize_name(audio_path.stem) + "_transcript.txt"
        out_path = OUTPUT_DIR / out_name
        tmp_path = OUTPUT_DIR / (out_name + ".tmp")

        # 파일명 정보 파싱 (정보용)
        info = parse_filename_safe(audio_path.name)

        start = time.time()

        try:
            segments, seg_info = model.transcribe(
                str(audio_path), language=LANGUAGE,
                beam_size=5, vad_filter=True,
                vad_parameters=dict(min_silence_duration_ms=500)
            )

            all_segments = []
            lines = []
            for seg in segments:
                h, m, s = int(seg.start//3600), int((seg.start%3600)//60), int(seg.start%60)
                all_segments.append({'start': seg.start, 'end': seg.end, 'text': seg.text.strip()})
                lines.append(f"[{h:02d}:{m:02d}:{s:02d}] {seg.text.strip()}")

            elapsed = time.time() - start
            dur = seg_info.duration / 60
            lec_num = info['lecture'] if info else '?'
            inst = info['instructor'] if info else '?'
            crs = info['course'] if info else '?'

            header = (
                f"# 전사문: {audio_path.stem}\n\n"
                f"- **원본**: {audio_path.name}\n"
                f"- **강사**: {inst}\n"
                f"- **강의**: {crs}\n"
                f"- **회차**: {lec_num}회\n"
                f"- **길이**: {dur:.1f}분\n"
                f"- **모델**: faster-whisper {MODEL_SIZE}\n"
                f"- **전사일**: {datetime.now().strftime('%Y-%m-%d %H:%M')}\n"
                f"- **소요시간**: {elapsed:.1f}초\n\n---\n\n"
            )

            # 임시저장 -> 완료 -> rename
            with open(tmp_path, 'w', encoding='utf-8') as f:
                f.write(header + '\n'.join(lines))
            tmp_path.rename(out_path)

            # 원본 녹음 파일 이동 완료 처리
            import shutil
            done_path = DONE_DIR / audio_path.name
            shutil.move(str(audio_path), str(done_path))

            results.append({'file': audio_path.name, 'status': 'success'})
            print(f"  ✅ {dur:.1f}분 → {elapsed:.1f}초 ({len(all_segments)} seg) [원본 이동 완료]")

        except Exception as e:
            print(f"  ❌ 오류: {e}")
            if tmp_path.exists():
                tmp_path.unlink()
            results.append({'file': audio_path.name, 'status': 'error', 'error': str(e)})

    ok = sum(1 for r in results if r['status']=='success')
    err = sum(1 for r in results if r['status']=='error')
    print(f"\n{'='*60}")
    print(f"🎉 완료: 성공 {ok} / 오류 {err}")
    print(f"{'='*60}")

    log = OUTPUT_DIR / f"transcribe_log_{datetime.now().strftime('%Y%m%d_%H%M')}.json"
    with open(log, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    print(f"\n📋 로그: {log.name}")

✅ 전사할 파일이 없습니다.


## 5️⃣ 다음 단계

전사 완료 후 로컬에서 후처리:
```powershell
python "h:\내 드라이브\.agent\skills\whisper-transcribe\scripts\post_transcribe.py"
```